In [1]:
import cv2
import numpy as np

cv2.namedWindow("Camera", cv2.WINDOW_NORMAL)

capture = cv2.VideoCapture(0)

def get_color(frame_bgr):
    bbox = cv2.selectROI("Camera", frame_bgr)
    x, y, w, h = map(int, bbox)
    roi = frame_bgr[y:y+h, x:x+w]
    hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    color = (np.median(hsv_roi[:,:,0]), np.median(hsv_roi[:,:,1]), np.median(hsv_roi[:,:,2]))
    cv2.destroyWindow("Color selection")
    return color


def get_ball(hsv_image, color):
    if color is None: return False, (-1, -1, -1, None)
    lower = (max(0, color[0] - 10), max(50, color[1] - 40), max(50, color[2] - 40))
    upper = (min(180, color[0] + 10), 255, 255)
    mask = cv2.inRange(hsv_image, lower, upper)
    mask = cv2.erode(mask, None, iterations=2)
    mask = cv2.dilate(mask, None, iterations=2)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) > 0:
        contour = max(contours, key=cv2.contourArea)
        (x, y), radius = cv2.minEnclosingCircle(contour)
        return True, (int(x), int(y), int(radius), mask)
    return False, (-1, -1, -1, mask)

target_color = None
path = []

while capture.isOpened():
    ret, frame = capture.read()
    if not ret: break

    blurred = cv2.GaussianBlur(frame, (11, 11), 0)
    hsv = cv2.cvtColor(blurred, cv2.COLOR_BGR2HSV)

    key = chr(cv2.waitKey(1) & 0xFF)

    if key == 'q':
        break
    if key == 'r':
        target_color = get_color(frame)
        path = []
    if key == 'w':
        path = []

    found, (x, y, radius, mask) = get_ball(hsv, target_color)

    if found:
        cv2.circle(frame, (x, y), radius, (0, 255, 0), 2)
        cv2.circle(frame, (x, y), 5, (0, 0, 255), -1)
        
        path.append(((x, y), radius))

    for i in range(1, len(path)):
        prev_pt, prev_r = path[i-1]
        curr_pt, curr_r = path[i]
        
        thickness = int(curr_r / 4) 
        thickness = max(1, thickness)
        
        cv2.line(frame, prev_pt, curr_pt, (0, 0, 255), thickness)

    cv2.imshow("Camera", frame)

capture.release()
cv2.destroyAllWindows()

Select a ROI and then press SPACE or ENTER button!
Cancel the selection process by pressing c button!
